# 🔍 MeatGuard-S — Inference & Explainability

**Phase 1 · RO3 (modules)** — Run a prediction and visualise the three XAI methods: **Grad-CAM**, **Occlusion Sensitivity**, and **SHAP**.

Mirrors `src/xai/gradcam.py`, `src/xai/occlusion.py`, `src/xai/shap_module.py`.

## 1. Setup & load model

In [ ]:
# Run once on a fresh Colab/Kaggle runtime (uncomment):
# !pip install torch torchvision timm einops scikit-learn matplotlib seaborn tqdm opencv-python shap scikit-image onnx

In [ ]:
import os, sys

# --- Path bootstrap: make ../src importable on local, Colab, and Kaggle ------
def add_src_to_path():
    here = os.getcwd()
    candidates = [
        os.path.abspath(os.path.join(here, "..", "src")),   # local: notebooks/ -> ../src
        os.path.abspath(os.path.join(here, "src")),         # run from repo root
        "/content/MeatGuard_II/src",                        # Colab (git clone target)
        "/kaggle/working/MeatGuard_II/src",                 # Kaggle
    ]
    for c in candidates:
        if os.path.isdir(c):
            if c not in sys.path:
                sys.path.insert(0, c)
            return c
    raise FileNotFoundError(
        "Could not locate the MeatGuard_II/src package. "
        "On Colab/Kaggle, clone the repo into /content (or /kaggle/working) first."
    )

SRC_DIR = add_src_to_path()
print("Using src package at:", SRC_DIR)

In [ ]:
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from model.architecture import MeatGuardS
from data.transforms import get_val_test_transform
from xai.gradcam import GradCAM
from xai.occlusion import OcclusionSensitivity

In [ ]:
# Placeholder dataset (drop-in stand-in for the full MeatScan set).
# Override with:  os.environ["MEATGUARD_DATA"] = "/path/to/your/data"
DATA_DIR = os.environ.get("MEATGUARD_DATA", r"D:/Research_Work/MeatGuard/data/test")
ARTIFACTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "artifacts"))
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("Data dir :", DATA_DIR)
print("Artifacts:", ARTIFACTS_DIR)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MeatGuardS(num_classes=2)

ckpt = os.path.join(ARTIFACTS_DIR, "meatguard_s_final.pth")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print("Loaded weights:", ckpt)
else:
    print("No checkpoint found — using randomly initialised weights (run notebook 01 first).")
model.eval().to(device)

## 2. Load a sample image

In [ ]:
candidates = sorted(glob.glob(os.path.join(DATA_DIR, "*", "*.jpg")) +
                    glob.glob(os.path.join(DATA_DIR, "*", "*.png")))
assert candidates, f"No images found under {DATA_DIR}"
img_path = candidates[0]
print("Sample:", img_path)

img = Image.open(img_path).convert("RGB")
transform = get_val_test_transform()
input_tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(input_tensor), dim=1)[0]
pred = int(probs.argmax())
print(f"Prediction: {['Fresh','Spoiled'][pred]}  (conf={probs[pred]:.3f})")

## 3. Grad-CAM

Class-discriminative heatmap from the last conv stage (`stage4`).

In [ ]:
gcam = GradCAM(model, target_layer="stage4")
cam_map = gcam.generate(input_tensor, class_idx=pred)
overlay = gcam.apply_overlay(input_tensor, cam_map)
gcam.remove_hooks()
print("Grad-CAM map shape:", cam_map.shape)

## 4. Occlusion Sensitivity

Model-agnostic confidence-drop map — a cross-check on Grad-CAM. (Coarse stride keeps it fast in a notebook.)

In [ ]:
occ = OcclusionSensitivity(model, patch_size=32, stride=24)
occ_map = occ.generate_map(input_tensor, class_idx=pred)
print("Occlusion map shape:", occ_map.shape)

## 5. SHAP (optional, heavier)

Superpixel attribution via `GradientExplainer`. Guarded because it depends on `shap`/`scikit-image` being installed and is slower than the other two.

In [ ]:
shap_overlay = None
try:
    from xai.shap_module import MeatGuardSHAP
    background = torch.stack([transform(Image.open(p).convert("RGB"))
                              for p in candidates[:8]]).to(device)
    explainer = MeatGuardSHAP(model, background)
    shap_values = explainer.explain(input_tensor, n_samples=50)
    print("SHAP values computed.")
except Exception as e:
    print(f"[shap] skipped: {e}")

## 6. Visualise

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img.resize((224, 224))); axes[0].set_title("Original")
axes[1].imshow(overlay);                axes[1].set_title("Grad-CAM")
axes[2].imshow(occ_map, cmap="jet");     axes[2].set_title("Occlusion")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()